# HMM et décodage de Viterbi — l'étiquetage morphosyntaxique probabiliste

**Navigation** : [<< Précédent — 04. Grammaires PCFG et parsing CYK](04_PCFG_CYK_Parsing.ipynb) | [Index de la série](README.md) | **05 (ce notebook)** | [Suivant — 06. Recherche d'information (TF-IDF, BM25) >>](06-Recherche-Information-TFIDF-BM25.ipynb)

**Sommaire** : [1. Le corpus](#1-le-corpus--un-mini-pos-francais-synthetique) · [2. Le modèle](#2-le-hmm--deux-distributions-et-une-hypothese-markovienne) · [3. Forward](#3-forward--la-vraisemblance-pw-en-log-espace) · [4. Viterbi](#4-viterbi-et-backpointers--le-chemin-le-plus-probable) · [5. Le contexte renverse](#5-le-cas-non-degenere--la-ou-le-contexte-renverse-la-decision) · [6. Baseline](#6-la-baseline-independante-par-token--le-plancher-a-battre) · [7. Témoin hmmlearn](#7-temoin-bibliotheque-i--hmmlearn-les-memes-parametres-recharges) · [8. Témoin NLTK](#8-temoin-bibliotheque-ii--nltk-un-pipeline-complet-sans-lissage) · [9. Analyse d'erreurs](#9-analyse-derreurs--six-tokens-resistent) · [Exercices](#exercice-1--backward-et-decodage-marginal-mpm) · [Conclusion](#à-retenir)

Le [02_NGrammes_Modeles_De_Langue](02_NGrammes_Modeles_De_Langue.ipynb) a modélisé `P(phrase)` en comptant les **mots observés**. Ce notebook franchit l'étape historique suivante : et si ce qui intéresse — ici, la **catégorie grammaticale** de chaque mot — n'est **pas observé** ? Un modèle de Markov caché (HMM) suppose que la phrase est **générée** par une chaîne d'étiquettes invisibles : une chaîne de Markov d'états (les catégories) émet des symboles observés (les mots). Étiqueter, c'est alors **retrouver la chaîne d'états la plus probable** ayant pu produire la phrase — par programmation dynamique, avec l'algorithme de **Viterbi** et ses backpointers.

**Le fil conducteur** : le français regorge de mots ambigus — *la porte verte* (nom) contre *elle porte une robe* (verbe), et même *le matelot voile la voile* où les deux lectures cohabitent dans la même phrase. Un décideur token par token ne peut pas les départager ; une chaîne de Markov sur les étiquettes le peut, parce que `DET → NOM` et `PRON → VER` sont des transitions de probabilités très différentes. Ce notebook **mesure** ce renversement, token par token.

**Approche** : tout est construit from scratch sur CPU — estimation MLE des transitions et émissions, lissage add-1, forward en log-espace, Viterbi avec backpointers. Chaque algorithme est ensuite confronté à un **témoin bibliothèque réellement exécuté** : `hmmlearn` recharge **les mêmes paramètres** et doit reproduire forward et Viterbi à l'identique ; `nltk` entraîne son propre HMM supervisé **sans lissage** — et son effondrement mesuré sur les mots inconnus justifie le nôtre.

**Position dans la série** : le HMM est le modèle **génératif** de l'étiquetage séquentiel ; le [03_CRF_Etiquetage_Sequentiel](03_CRF_Etiquetage_Sequentiel.ipynb) en montrera le pendant **discriminatif** (CRF linéaire, Lafferty et al. 2001), qui apprend les mêmes transitions mais conditionnellement aux mots. Les deux notebooks partagent le même plan d'expérience : baseline token-wise, témoin bibliothèque, analyse d'erreurs.
**Version jumelle** : [Python](05-HMM-Viterbi.ipynb).

**Interopérabilité** : installer CPython, NumPy, hmmlearn et NLTK. Avant Jupyter, définir `PYTHONNET_PYDLL` vers la bibliothèque partagée Python et au besoin `PYTHONHOME` et `PYTHONPATH` pour sélectionner l’environnement. Python.NET ne sert qu’aux deux témoins ; les calculs principaux sont en C#.


## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :

1. **Formuler** l'étiquetage morphosyntaxique comme inférence dans un modèle de Markov caché : états, émissions, hypothèse markovienne ;
2. **Estimer** les paramètres d'un HMM par maximum de vraisemblance sur un corpus annoté, avec lissage add-1 et symbole d'inconnu ;
3. **Implémenter** l'algorithme forward en log-espace (vraisemblance `log P(W)`) et le décodage de Viterbi avec backpointers (chemin le plus probable) ;
4. **Mesurer** l'apport du contexte de séquence : là où le renversement de décision par rapport à un décideur token par token se produit, et ce qu'il corrige ;
5. **Confronter** l'implémentation à deux bibliothèques exécutées (`hmmlearn` en concordance exacte, `nltk` sans lissage en contre-exemple mesuré).

### Prérequis

- C# et probabilités ; CPython avec NumPy, hmmlearn et NLTK pour les témoins
- Avoir parcouru le [02_NGrammes_Modeles_De_Langue](02_NGrammes_Modeles_De_Langue.ipynb) (MLE, lissage, perplexité)

### Durée estimée : 55 minutes

## 1. Le corpus — un mini-POS français synthétique

Le corpus embarqué ci-dessous est **synthétique** : il a été **rédigé à la main pour ce notebook** (dépôt CoursIA, issue #17695), dans le prolongement du registre du [03_CRF_Etiquetage_Sequentiel](03_CRF_Etiquetage_Sequentiel.ipynb).

> **Provenance et statut** — Corpus **synthétique, écrit pour ce notebook** ; il **n'est pas** un corpus de référence (ni Universal Dependencies, ni French Treebank) et ses statistiques n'ont valeur que pédagogique. Il est conçu pour exposer de façon auditable trois phénomènes : (a) des mots ambigus **NOM/VER** (*porte*, *voile*, *coupe*, *garde*, *avance*), (b) des clitiques **DET/PRON** (*le*, *la*, *les*), (c) une phrase-test où **les deux lectures d'un même mot cohabitent** (*le matelot voile la voile*). Licence : celle du dépôt.

Le jeu d'étiquettes compte 8 catégories : `DET`, `NOM`, `VER`, `ADJ`, `PREP`, `PRON`, `ADV`, `PUNCT` — un POS simplifié, volontairement resserré pour que le treillis de la section 4 reste lisible cellule par cellule.

In [1]:
#r "nuget: pythonnet, 3.0.5"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using System.Text;
using System.Net;
using Microsoft.DotNet.Interactive.Formatting;
using Python.Runtime;
record PlotSvg(string Content);
Formatter.Register<PlotSvg>((v,w)=>w.Write(v.Content), "text/html");
string F(double x) => x.ToString("0.###",CultureInfo.InvariantCulture);
string E(string s) => WebUtility.HtmlEncode(s);
string Text(double x,double y,string t,string anchor="middle") => $"<text x='{F(x)}' y='{F(y)}' text-anchor='{anchor}' font-size='12' fill='#222'>{E(t)}</text>";
string StartSvg(string title) => "<svg xmlns='http://www.w3.org/2000/svg' width='760' height='380' viewBox='0 0 760 380'><rect width='760' height='380' fill='white'/>"+Text(380,24,title);
void Bars(string title,string ylabel,string[] labels,double[][] series,string[] legends,string[] colors,double ymax) {
    var svg=new StringBuilder(StartSvg(title)); int n=labels.Length; double left=76,top=50,width=640,height=240;
    for(int j=0;j<=5;j++) { double y=top+height-j*height/5; svg.Append($"<line x1='{F(left)}' y1='{F(y)}' x2='{F(left+width)}' y2='{F(y)}' stroke='#ddd'/>"); svg.Append(Text(left-8,y+4,(ymax*j/5).ToString("0.###",CultureInfo.InvariantCulture),"end")); }
    svg.Append($"<path d='M {F(left)} {F(top)} V {F(top+height)} H {F(left+width)}' fill='none' stroke='black'/>");
    double group=width/n, bw=Math.Min(30,group/(series.Length+1));
    for(int i=0;i<n;i++) { double cx=left+(i+.5)*group; svg.Append(Text(cx,top+height+18,labels[i]));
        for(int j=0;j<series.Length;j++) { double val=series[j][i], bh=height*val/ymax, x=cx+(j-(series.Length-1)/2.0)*bw-bw/2;
            svg.Append($"<rect x='{F(x)}' y='{F(top+height-bh)}' width='{F(bw-2)}' height='{F(bh)}' fill='{colors[j]}'/>"); }
    }
    svg.Append(Text(left,42,ylabel,"start"));
    for(int j=0;j<legends.Length;j++) { double x=90+j*210; svg.Append($"<rect x='{F(x)}' y='338' width='14' height='14' fill='{colors[j]}'/>"); svg.Append(Text(x+20,350,legends[j],"start")); }
    svg.Append("</svg>"); display(new PlotSvg(svg.ToString()));
}
if (!PythonEngine.IsInitialized) {
    var dll=Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");
    if (string.IsNullOrWhiteSpace(dll)) throw new InvalidOperationException("Définir PYTHONNET_PYDLL avant de lancer Jupyter.");
    Runtime.PythonDLL=dll; PythonEngine.Initialize(); PythonEngine.BeginAllowThreads();
}
using(Py.GIL()) {
    dynamic sys=Py.Import("sys"); dynamic np=Py.Import("numpy"); dynamic hm=Py.Import("hmmlearn"); dynamic nltk=Py.Import("nltk");
    Console.WriteLine($".NET {Environment.Version} | Python.NET {typeof(PythonEngine).Assembly.GetName().Version}");
    Console.WriteLine($"Python {sys.version} | NumPy {np.__version__} | hmmlearn {hm.__version__} | NLTK {nltk.__version__}");
}

Installed Packages pythonnet, 3.0.5

.NET 10.0.14 | Python.NET 3.0.5.0


Python 3.13.16 (main, Oct 14 2026, 01:04:01) [Clang 23.1.3 ] | NumPy 2.5.4 | hmmlearn 0.3.3 | NLTK 3.10.4


### Lecture — l'environnement réellement exécuté

La cellule ci-dessus confirme les **versions effectives** de l'environnement d'exécution — elles sont la source de vérité pour ce notebook (aucune version n'est épinglée à la main dans la prose). Tout ce qui suit tourne sur CPU, sans hasard : **aucune graine n'est nécessaire**, l'estimation MLE et les deux algorithmes de programmation dynamique sont déterministes.

In [2]:
// Corpus POS français synthétique, rédigé pour ce carnet.
var corpus = new (string Word,string Tag)[][] {
    new (string Word,string Tag)[] { ("la", "DET"), ("porte", "NOM"), ("verte", "ADJ"), ("est", "VER"), ("fermée", "ADJ"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("une", "DET"), ("voile", "NOM"), ("blanche", "ADJ"), ("glisse", "VER"), ("sur", "PREP"), ("la", "DET"), ("mer", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("la", "DET"), ("coupe", "NOM"), ("dorée", "ADJ"), ("brille", "VER"), ("sur", "PREP"), ("la", "DET"), ("table", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("la", "DET"), ("garde", "NOM"), ("royale", "ADJ"), ("dort", "VER"), ("encore", "ADV"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("une", "DET"), ("avance", "NOM"), ("sérieuse", "ADJ"), ("surprend", "VER"), ("la", "DET"), ("banque", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("le", "DET"), ("voile", "NOM"), ("noir", "ADJ"), ("recouvre", "VER"), ("la", "DET"), ("fenêtre", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("la", "DET"), ("porte", "NOM"), ("grince", "VER"), ("dans", "PREP"), ("la", "DET"), ("maison", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("leur", "DET"), ("garde", "NOM"), ("veille", "VER"), ("toute", "DET"), ("la", "DET"), ("nuit", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("il", "PRON"), ("porte", "VER"), ("un", "DET"), ("manteau", "NOM"), ("gris", "ADJ"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("elle", "PRON"), ("voile", "VER"), ("la", "DET"), ("fenêtre", "NOM"), ("sale", "ADJ"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("il", "PRON"), ("coupe", "VER"), ("la", "DET"), ("pomme", "NOM"), ("rouge", "ADJ"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("il", "PRON"), ("garde", "VER"), ("la", "DET"), ("clé", "NOM"), ("précieuse", "ADJ"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("elle", "PRON"), ("avance", "VER"), ("doucement", "ADV"), ("dans", "PREP"), ("le", "DET"), ("brouillard", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("la", "DET"), ("femme", "NOM"), ("porte", "VER"), ("un", "DET"), ("chapeau", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("le", "DET"), ("vent", "NOM"), ("voile", "VER"), ("les", "DET"), ("toits", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("le", "DET"), ("cuisinier", "NOM"), ("coupe", "VER"), ("le", "DET"), ("pain", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("la", "DET"), ("nourrice", "NOM"), ("garde", "VER"), ("les", "DET"), ("enfants", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("la", "DET"), ("procession", "NOM"), ("avance", "VER"), ("lentement", "ADV"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("il", "PRON"), ("le", "PRON"), ("garde", "VER"), ("précieusement", "ADV"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("elle", "PRON"), ("la", "PRON"), ("porte", "VER"), ("toujours", "ADV"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("on", "PRON"), ("les", "PRON"), ("coupe", "VER"), ("chaque", "DET"), ("matin", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("il", "PRON"), ("la", "PRON"), ("voile", "VER"), ("encore", "ADV"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("le", "DET"), ("chat", "NOM"), ("dort", "VER"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("la", "DET"), ("souris", "NOM"), ("mange", "VER"), ("sans", "PREP"), ("bruit", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("un", "DET"), ("homme", "NOM"), ("lit", "VER"), ("le", "DET"), ("journal", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("elle", "PRON"), ("chante", "VER"), ("dans", "PREP"), ("la", "DET"), ("cuisine", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("le", "DET"), ("facteur", "NOM"), ("ouvre", "VER"), ("la", "DET"), ("boîte", "NOM"), ("bleue", "ADJ"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("ils", "PRON"), ("observent", "VER"), ("la", "DET"), ("scène", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("la", "DET"), ("lumière", "NOM"), ("éclaire", "VER"), ("le", "DET"), ("jardin", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("un", "DET"), ("enfant", "NOM"), ("dessine", "VER"), ("sur", "PREP"), ("une", "DET"), ("page", "NOM"), ("blanche", "ADJ"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("elle", "PRON"), ("ferme", "VER"), ("la", "DET"), ("fenêtre", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("le", "DET"), ("journal", "NOM"), ("annonce", "VER"), ("une", "DET"), ("réponse", "NOM"), ("claire", "ADJ"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("la", "DET"), ("voile", "NOM"), ("rouge", "ADJ"), ("tourne", "VER"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("elle", "PRON"), ("porte", "VER"), ("une", "DET"), ("robe", "NOM"), ("bleue", "ADJ"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("il", "PRON"), ("voile", "VER"), ("la", "DET"), ("vérité", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("le", "DET"), ("garde", "NOM"), ("ferme", "VER"), ("la", "DET"), ("grille", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("elle", "PRON"), ("coupe", "VER"), ("le", "DET"), ("gazon", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("la", "DET"), ("coupe", "NOM"), ("brille", "VER"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("il", "PRON"), ("la", "PRON"), ("garde", "VER"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("une", "DET"), ("avance", "NOM"), ("rapide", "ADJ"), ("surprend", "VER"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("le", "DET"), ("matelot", "NOM"), ("voile", "VER"), ("la", "DET"), ("voile", "NOM"), (".", "PUNCT") },
    new (string Word,string Tag)[] { ("le", "DET"), ("concierge", "NOM"), ("ouvre", "VER"), ("la", "DET"), ("porte", "NOM"), ("verte", "ADJ"), (".", "PUNCT") }
};
Console.WriteLine($"Corpus : {corpus.Length} phrases, {corpus.Sum(s=>s.Length)} tokens, {corpus.SelectMany(s=>s).Select(x=>x.Word).Distinct().Count()} types, {corpus.SelectMany(s=>s).Select(x=>x.Tag).Distinct().Count()} étiquettes");

Corpus : 42 phrases, 250 tokens, 98 types, 8 étiquettes


### Lecture — le corpus synthétique en chiffres

**42 phrases, 250 tokens, 98 types, 8 étiquettes** (sortie ci-dessus) : c'est un corpus miniature délibéré — assez petit pour auditer chaque phrase, assez structuré pour que les comptages MLE portent. Les phrases sont organisées en quatre blocs visibles dans le code : contextes **NOM** des mots ambigus, contextes **VER**, phrases à **clitiques** (*il le garde* : `le` est alors pronom), et phrases de fond non ambiguës. Les dix dernières phrases constituent le **test** — choisies pour placer certains mots ambigus dans leur contexte **minoritaire** du train (*la voile*, *le garde*, *une avance*, *la coupe*, *la porte*, et *la* pronom : 7 des 22 occurrences d'ambigus y sont minoritaires, les autres restent en contexte majoritaire), et pour contenir des mots inconnus (*robe*, *matelot*, *vérité*…).

In [3]:
var train=corpus.Take(corpus.Length-10).ToArray(); var test=corpus.Skip(corpus.Length-10).ToArray();
string[] tags={"DET","NOM","VER","ADJ","PREP","PRON","ADV","PUNCT"};
var tagIndex=tags.Select((x,i)=>(x,i)).ToDictionary(x=>x.x,x=>x.i); int K=tags.Length;
var trainWords=train.SelectMany(x=>x).Select(x=>x.Word).ToHashSet();
var ambiguous=corpus.SelectMany(x=>x).GroupBy(x=>x.Word).Where(g=>g.Select(x=>x.Tag).Distinct().Count()>1).Select(g=>g.Key).OrderBy(x=>x,StringComparer.Ordinal).ToArray();
var unknown=test.SelectMany(x=>x).Select(x=>x.Word).Where(x=>!trainWords.Contains(x)).Distinct().OrderBy(x=>x,StringComparer.Ordinal).ToArray();
var dist=tags.Select(t=>(double)train.SelectMany(x=>x).Count(x=>x.Tag==t)).ToArray();
Console.WriteLine($"TRAIN : {train.Length} phrases / {train.Sum(x=>x.Length)} tokens ; TEST : {test.Length} phrases / {test.Sum(x=>x.Length)} tokens");
Console.WriteLine($"Mots ambigus ({ambiguous.Length}) : {string.Join(", ",ambiguous)}");
Console.WriteLine($"Inconnus du train ({unknown.Length}) : {string.Join(", ",unknown)}");
Console.WriteLine("Distribution TRAIN : "+string.Join(", ",tags.Select((t,i)=>$"{t}={dist[i]}")));
Bars("Distribution des étiquettes du train", "occurrences",tags,new[]{dist},new[]{"Train"},new[]{"#1f77b4"},Math.Ceiling(dist.Max()/10)*10);

TRAIN : 32 phrases / 197 tokens ; TEST : 10 phrases / 53 tokens


Mots ambigus (8) : avance, coupe, garde, la, le, les, porte, voile


Inconnus du train (8) : concierge, gazon, grille, matelot, rapide, robe, tourne, vérité


Distribution TRAIN : DET=45, NOM=45, VER=32, ADJ=14, PREP=7, PRON=16, ADV=6, PUNCT=32


Distribution des étiquettes du train 0 10 20 30 40 50 DET NOM VER ADJ PREP PRON ADV PUNCT occurrences Train

### Lecture — ambiguïté et inconnus, les deux ennemis mesurés

La sortie ci-dessus liste **8 mots ambigus** : les cinq NOM/VER (*avance, coupe, garde, porte, voile*) et les trois clitiques DET/PRON (*la, le, les*). Elle liste aussi **8 types inconnus** du train présents dans le test — c'est le second problème : un MLE brut donnerait `P(w|t) = 0` pour chacun, donc une vraisemblance nulle sur huit des dix phrases de test. Ces deux listes pilotent tout le notebook : l'ambiguïté justifie Viterbi (sections 4–5), les inconnus justifient le lissage (section 2) et expliquent l'effondrement du témoin NLTK (section 8). La distribution du train — `DET` et `NOM` largement en tête — est visible sur la figure : c'est un français de phrases simples, avec un verbe par phrase.

## 2. Le HMM — deux distributions et une hypothèse markovienne

Un HMM d'étiquetage suppose que la phrase annotée $(w_1, \dots, w_T, t_1, \dots, t_T)$ est générée ainsi :

$$P(T, W) \;=\; \pi(t_1) \cdot B_{t_1}(w_1) \cdot \prod_{i=2}^{T} A_{t_{i-1}, t_i} \cdot B_{t_i}(w_i)$$

où — avec $K = 8$ états et un vocabulaire $V$ :

- $\pi$ est la distribution des **étiquettes initiales** $P(t_1)$ ;
- $A \in \mathbb{R}^{K \times K}$ est la **matrice de transitions** $P(t_i \mid t_{i-1})$ — l'hypothèse markovienne : une étiquette ne dépend que de la **précédente** ;
- $B \in \mathbb{R}^{K \times V}$ est la matrice d'**émissions** $P(w_i \mid t_i)$ — chaque état tire son mot.

Sur un corpus annoté, l'estimation par maximum de vraisemblance est un simple **comptage** : chaque occurrence de bigramme d'étiquettes incrémente $A$, chaque paire (étiquette, mot) incrémente $B$. Le piège est elsewhere : les zéros. Une transition jamais vue, un mot jamais émis par une étiquette, un mot **inconnu** du train — chacun vaut $\log 0 = -\infty$ et tue la suite. Nous appliquons le lissage **add-1** (Laplace) des deux côtés, avec un symbole d'inconnu `<UNK>` ajouté au vocabulaire — exactement le remède du [02_NGrammes_Modeles_De_Langue](02_NGrammes_Modeles_De_Langue.ipynb), transposé aux étiquettes.

In [4]:
var vocab=trainWords.OrderBy(x=>x,StringComparer.Ordinal).Concat(new[]{"<UNK>"}).ToArray();
var wordIndex=vocab.Select((x,i)=>(x,i)).ToDictionary(x=>x.x,x=>x.i); int V=vocab.Length;
int Wi(string w) => wordIndex.GetValueOrDefault(w,wordIndex["<UNK>"]);
var pi=Enumerable.Repeat(1.0,K).ToArray(); var A=new double[K,K]; var B=new double[K,V];
for(int i=0;i<K;i++) { for(int j=0;j<K;j++) A[i,j]=1; for(int j=0;j<V;j++) B[i,j]=1; }
foreach(var sent in train) for(int i=0;i<sent.Length;i++) { int t=tagIndex[sent[i].Tag]; B[t,Wi(sent[i].Word)]++; if(i==0) pi[t]++; else A[tagIndex[sent[i-1].Tag],t]++; }
double piSum=pi.Sum(); for(int i=0;i<K;i++) { pi[i]/=piSum; double sa=0,sb=0; for(int j=0;j<K;j++) sa+=A[i,j]; for(int j=0;j<V;j++) sb+=B[i,j]; for(int j=0;j<K;j++) A[i,j]/=sa; for(int j=0;j<V;j++) B[i,j]/=sb; }
var logPi=pi.Select(x=>Math.Log(x)).ToArray(); var logA=new double[K,K]; var logB=new double[K,V];
for(int i=0;i<K;i++) { for(int j=0;j<K;j++) logA[i,j]=Math.Log(A[i,j]); for(int j=0;j<V;j++) logB[i,j]=Math.Log(B[i,j]); }
if(Math.Abs(pi.Sum()-1)>1e-12 || Enumerable.Range(0,K).Any(i=>Math.Abs(Enumerable.Range(0,K).Sum(j=>A[i,j])-1)>1e-12 || Math.Abs(Enumerable.Range(0,V).Sum(j=>B[i,j])-1)>1e-12)) throw new Exception("Normalisation HMM incorrecte");
Console.WriteLine($"Paramètres pi ({K}), A ({K}x{K}), B ({K}x{V}, dont UNK)");
Console.WriteLine($"P(NOM|DET)={A[tagIndex["DET"],tagIndex["NOM"]]:F3} ; P(VER|PRON)={A[tagIndex["PRON"],tagIndex["VER"]]:F3} ; P(UNK|ADV)={B[tagIndex["ADV"],Wi("<UNK>")]:F4}");

Paramètres pi (8), A (8x8), B (8x91, dont UNK)


P(NOM|DET)=0.849 ; P(VER|PRON)=0.542 ; P(UNK|ADV)=0.0103


### Lecture — trois tables, et un effet de bord du lissage

Les transitions racontent la grammaire du corpus : `P(NOM | DET)` et `P(VER | PRON)` (sortie ci-dessus) sont les deux colonnes qui feront tout le travail de la section 5 — un déterminant appelle un nom, un pronom appelle un verbe. L'effet de bord visible sur la troisième ligne est un **piège classique du add-1** : l'étiquette la plus rare (`ADV` ici) reçoit la probabilité d'inconnu **la plus élevée**, puisque le lissage répartit la masse $\alpha$ sur un dénominateur plus petit. Conséquence mesurée en section 9 : un mot inconnu **verbal** (*tourne*) sera d'abord attiré par `ADV` au niveau des émissions — le décodage de séquence le classe pourtant `PUNCT` au lieu de `VER`, comme le montre l'analyse d'erreurs en section 9.

In [5]:
string[] ambigusNV={"porte","voile","coupe","garde","avance"};
var pn=ambigusNV.Select(w=>B[tagIndex["NOM"],Wi(w)]).ToArray(); var pv=ambigusNV.Select(w=>B[tagIndex["VER"],Wi(w)]).ToArray();
for(int i=0;i<ambigusNV.Length;i++) Console.WriteLine($"{ambigusNV[i],-8} P(w|NOM)={pn[i]:F4} P(w|VER)={pv[i]:F4} rapport={pv[i]/pn[i]:F2}");
foreach(string w in new[]{"le","la","les"}) Console.WriteLine($"{w,-8} P(w|DET)={B[tagIndex["DET"],Wi(w)]:F4} P(w|PRON)={B[tagIndex["PRON"],Wi(w)]:F4}");
Bars("Émissions des cinq mots ambigus", "P(mot|tag)",ambigusNV,new[]{pn,pv},new[]{"P(mot|NOM)","P(mot|VER)"},new[]{"#1f77b4","#d62728"},Math.Ceiling(Math.Max(pn.Max(),pv.Max())*100)/100);

porte    P(w|NOM)=0.0221 P(w|VER)=0.0325 rapport=1.47


voile    P(w|NOM)=0.0221 P(w|VER)=0.0325 rapport=1.47


coupe    P(w|NOM)=0.0147 P(w|VER)=0.0325 rapport=2.21


garde    P(w|NOM)=0.0221 P(w|VER)=0.0325 rapport=1.47


avance   P(w|NOM)=0.0147 P(w|VER)=0.0244 rapport=1.66


le       P(w|DET)=0.0809 P(w|PRON)=0.0187


la       P(w|DET)=0.1691 P(w|PRON)=0.0280


les      P(w|DET)=0.0221 P(w|PRON)=0.0187


Émissions des cinq mots ambigus 0 0.008 0.016 0.024 0.032 0.04 porte voile coupe garde avance P(mot|tag) P(mot|NOM) P(mot|VER)

### Lecture — les émissions ordonnent, mais ne décident pas

Pour chacun des cinq mots, l'émission **verbale** domine la nominale dans le train (rapports supérieurs à 1 dans la sortie) : *porte* apparaît plus souvent comme verbe. Un décideur qui ne regarde que `P(w|t)` — qui coïncide ici avec le tag majoritaire pour ces mots — choisit ici **VER**, et se trompera sur *la porte verte*. La figure montre l'écart : réel, mais **modeste** (l'émission nominale reste du même ordre de grandeur). C'est précisément cette marge étroite qui rend le renversement par contexte possible : une transition `DET → NOM` massive suffit à la faire basculer. Les clitiques sont dans le même cas, `P(la|DET)` dominant `P(la|PRON)` — la section 5 montrera le cas *il la garde*, où les deux ambiguïtés se superposent.

## 3. Forward — la vraisemblance P(W) en log-espace

Avant de décoder, mesurons : quelle probabilité le modèle accorde-t-il à une **phrase** ? L'algorithme forward calcule

$$\log P(W) \;=\; \log \sum_{t_1, \dots, t_T} P(T, W),$$

la somme sur **tous les chemins d'étiquettes** possibles — à distinguer soigneusement du **maximum** de Viterbi (section 4). La récurrence, en log-espace pour éviter l'underflow ($K^T$ chemins, chacun un produit de $2T$ probabilités) :

$$\alpha_1(k) = \log\pi_k + \log B_k(w_1), \qquad \alpha_i(k) = \log \sum_j e^{\,\alpha_{i-1}(j) + \log A_{jk}} + \log B_k(w_i),$$

réalisée avec `LogSumExp` — le compagnon exact de la somme en log-espace. La table $\alpha$ cumule des **sommes partielles** : à ne pas confondre avec la table $\delta$ du **max** que Viterbi remplira sur le même treillis (section 4). C'est la même table triangulaire que la log-partition du CRF du [03_CRF_Etiquetage_Sequentiel](03_CRF_Etiquetage_Sequentiel.ipynb), et la même famille que la vraisemblance n-gramme du [02_NGrammes_Modeles_De_Langue](02_NGrammes_Modeles_De_Langue.ipynb) — mais marginalisée sur des états **cachés** au lieu d'être conditionnée aux mots observés.

In [6]:
double LogSumExp(IEnumerable<double> values) { var a=values.ToArray(); if(a.Length==0) return double.NegativeInfinity; double m=a.Max(); return double.IsNegativeInfinity(m)?m:m+Math.Log(a.Sum(x=>Math.Exp(x-m))); }
double Forward(int[] words,double[] lp,double[,] la,double[,] lb) {
    if(words.Length==0) return 0.0; int k=lp.Length; var prev=Enumerable.Range(0,k).Select(i=>lp[i]+lb[i,words[0]]).ToArray();
    for(int t=1;t<words.Length;t++) { int tt=t; var next=new double[k]; for(int j=0;j<k;j++) { int jj=j; next[j]=LogSumExp(Enumerable.Range(0,k).Select(i=>prev[i]+la[i,jj]))+lb[j,words[tt]]; } prev=next; }
    return LogSumExp(prev);
}
double ForwardWords(string[] ws) => Forward(ws.Select(Wi).ToArray(),logPi,logA,logB);
if(!double.IsNegativeInfinity(LogSumExp(new[]{double.NegativeInfinity,double.NegativeInfinity})) || ForwardWords(Array.Empty<string>())!=0) throw new Exception("Contrats somme vide / séquence vide");
for(int i=0;i<test.Length;i++) { var words=test[i].Select(x=>x.Word).ToArray(); Console.WriteLine($"test[{i}] {string.Join(" ",words),-40} logP={ForwardWords(words):F3}"); }
Console.WriteLine("Séquence vide : log P=0 ; logsumexp de valeurs toutes -inf : -inf");

test[0] la voile rouge tourne .                  logP=-18.490


test[1] elle porte une robe bleue .              logP=-23.731


test[2] il voile la vérité .                     logP=-17.670


test[3] le garde ferme la grille .               logP=-21.789


test[4] elle coupe le gazon .                    logP=-18.357


test[5] la coupe brille .                        logP=-14.061


test[6] il la garde .                            logP=-14.416


test[7] une avance rapide surprend .             logP=-20.394


test[8] le matelot voile la voile .              logP=-21.228


test[9] le concierge ouvre la porte verte .      logP=-26.574


Séquence vide : log P=0 ; logsumexp de valeurs toutes -inf : -inf


### Lecture — des vraisemblances finies, grâce au lissage

Chaque phrase du test reçoit une vraisemblance **finie** (aucun `-inf` dans la sortie) : c'est le lissage add-1 en acte — sans lui, chaque phrase contenant un inconnu vaudrait $\log 0$. Les phrases courtes et attendues (*il la garde*.) sont mieux loties que les longues avec inconnus (*le concierge ouvre la porte verte*.) : chaque token ajoute un facteur de probabilité < 1. Cette quantité servira de **jauge** au témoin `hmmlearn` (section 7) : sa méthode `score` doit reproduire ces valeurs à la précision machine.

## 4. Viterbi et backpointers — le chemin le plus probable

Décoder, c'est passer de la **somme** au **max** : quel chemin d'étiquettes $T^*$ maximise $P(T \mid W)$ ?

$$\delta_i(k) = \max_j \big[\, \delta_{i-1}(j) + \log A_{jk} \,\big] + \log B_k(w_i), \qquad \mathrm{bp}_i(k) = \arg\max_j \big[\, \delta_{i-1}(j) + \log A_{jk} \,\big].$$

La table $\delta$ se remplit comme la table $\alpha$ du forward, sur le même treillis ; la différence est le `max` au lieu de la somme, et le **backpointer** $\mathrm{bp}$ qui mémorise, pour chaque état au temps $i$, l'état prédécesseur gagnant. Le chemin se reconstruit **à l'envers** : état final $= \arg\max_k \delta_T(k)$, puis on suit les backpointers jusqu'au début. Coût : $O(T \cdot K^2)$ — le même treillis que forward, 8 états, une dizaine de positions : tout tient dans une figure.

In [7]:
(int[],double[,],int[]) Viterbi(int[] words,double[] lp,double[,] la,double[,] lb) {
    int n=words.Length,k=lp.Length; if(n==0) return (Array.Empty<int>(),new double[0,k],Array.Empty<int>());
    var d=new double[n,k]; var back=new int[n*k]; for(int j=0;j<k;j++) d[0,j]=lp[j]+lb[j,words[0]];
    for(int t=1;t<n;t++) for(int j=0;j<k;j++) { int best=0; for(int i=1;i<k;i++) if(d[t-1,i]+la[i,j]>d[t-1,best]+la[best,j]) best=i; back[t*k+j]=best; d[t,j]=d[t-1,best]+la[best,j]+lb[j,words[t]]; }
    int last=0; for(int j=1;j<k;j++) if(d[n-1,j]>d[n-1,last]) last=j; var path=new int[n]; path[n-1]=last;
    for(int t=n-1;t>0;t--) path[t-1]=back[t*k+path[t]]; return (path,d,back);
}
(int[],double[,],int[]) Decode(string[] ws) => Viterbi(ws.Select(Wi).ToArray(),logPi,logA,logB);
// Oracle indépendant : énumération des quatre chemins du petit HMM.
var op=new[]{Math.Log(.6),Math.Log(.4)}; var oa=new double[,]{{Math.Log(.999),Math.Log(.001)},{Math.Log(.001),Math.Log(.999)}};
var ob=new double[,]{{Math.Log(.9),Math.Log(.01)},{Math.Log(.8),Math.Log(.2)}};
var probs=new List<double>(); var paths=new List<(int,int)>();
for(int a=0;a<2;a++) for(int b=0;b<2;b++) { paths.Add((a,b)); probs.Add(Math.Exp(op[a]+ob[a,0]+oa[a,b]+ob[b,1])); }
var (oPath,_,_)=Viterbi(new[]{0,1},op,oa,ob); int bestOracle=probs.IndexOf(probs.Max());
if(Math.Abs(Forward(new[]{0,1},op,oa,ob)-Math.Log(probs.Sum()))>1e-12 || !oPath.SequenceEqual(new[]{paths[bestOracle].Item1,paths[bestOracle].Item2}) || !oPath.SequenceEqual(new[]{1,1}) || Viterbi(Array.Empty<int>(),op,oa,ob).Item1.Length!=0) throw new Exception("Oracle HMM incorrect");
int correct=0,total=0; foreach(var sent in test) { var path=Decode(sent.Select(x=>x.Word).ToArray()).Item1; correct+=path.Where((x,i)=>tags[x]==sent[i].Tag).Count(); total+=sent.Length; }
Console.WriteLine($"Oracle 4 chemins : log somme={Math.Log(probs.Sum()):F12}, chemin={string.Join(",",oPath)} ; meilleur préfixe = 0");
Console.WriteLine($"Viterbi TEST : {correct}/{total} = {(double)correct/total:F3}");
foreach(int n in new[]{1,8}) { var sent=test[n]; var (path,d,_)=Decode(sent.Select(x=>x.Word).ToArray()); Console.WriteLine($"test[{n}] : {string.Join(" ",sent.Select(x=>x.Word))}\n or : {string.Join(" ",sent.Select(x=>x.Tag))}\n Viterbi : {string.Join(" ",path.Select(x=>tags[x]))} | logP chemin {d[sent.Length-1,path[^1]]:F3}"); }

Oracle 4 chemins : log somme=-2.667266287307, chemin=1,1 ; meilleur préfixe = 0


Viterbi TEST : 47/53 = 0.887


test[1] : elle porte une robe bleue .
 or : PRON VER DET NOM ADJ PUNCT
 Viterbi : PRON VER DET NOM ADJ PUNCT | logP chemin -24.669


test[8] : le matelot voile la voile .
 or : DET NOM VER DET NOM PUNCT
 Viterbi : DET NOM VER DET NOM PUNCT | logP chemin -21.764


### Lecture — le double sens résolu dans la même phrase

Sur la phrase à double lecture *le matelot voile la voile*., Viterbi rend `DET NOM VER DET NOM` : le **premier** *voile* (après un syntagme nominal sujet) est décodé verbe, le **second** (après le déterminant *la*) est décodé nom — exactement l'or. Aucun décideur token par token ne peut produire ce couple, puisqu'il voit deux fois le même mot et doit répondre deux étiquettes différentes : la décision vient de la **position dans la chaîne**, portée par les transitions. Le score agrégé du test est prometteur ; il sera comparé honnêtement aux baselines en section 6 et aux témoins en sections 7–8. La valeur `logP chemin` est le **max** de Viterbi — strictement inférieure au `log P(W)` de la section 3, qui somme les chemins : leur différence est le log du rapport entre la somme totale et la masse du meilleur chemin.

In [8]:
var ws=test[8].Select(x=>x.Word).ToArray(); var (chemin,dlt,bp)=Decode(ws);
Console.WriteLine($"Phrase : {string.Join(" ",ws)} ; Viterbi : {string.Join(" ",chemin.Select(i=>tags[i]))}");
for(int t=0;t<ws.Length;t++) { int tt=t; var best=Enumerable.Range(0,K).OrderByDescending(i=>dlt[tt,i]).Take(2); Console.WriteLine($"{ws[t],-9} "+string.Join(" | ",best.Select(i=>$"{tags[i]} {dlt[tt,i]:F2} <- {(tt==0?"—":tags[bp[tt*K+i]])}"))); }
var fig=new StringBuilder(StartSvg("Treillis Viterbi : écart au meilleur état par position"));
double x0=95,y0=55,cw=76,ch=27; double minDelta=0;
for(int t=0;t<ws.Length;t++) { double max=Enumerable.Range(0,K).Max(i=>dlt[t,i]); for(int k=0;k<K;k++) minDelta=Math.Min(minDelta,dlt[t,k]-max); }
for(int t=0;t<ws.Length;t++) { double max=Enumerable.Range(0,K).Max(i=>dlt[t,i]); fig.Append(Text(x0+(t+.5)*cw,y0+K*ch+20,ws[t]));
 for(int k=0;k<K;k++) { double r=(dlt[t,k]-max)/minDelta; int shade=(int)(240-185*r); string color=$"#{shade:X2}{shade:X2}FF"; fig.Append($"<rect x='{F(x0+t*cw)}' y='{F(y0+k*ch)}' width='{F(cw)}' height='{F(ch)}' fill='{color}' stroke='white'/>"); }
 fig.Append($"<circle cx='{F(x0+(t+.5)*cw)}' cy='{F(y0+(chemin[t]+.5)*ch)}' r='8' fill='none' stroke='#111' stroke-width='5'/>");
 fig.Append($"<circle cx='{F(x0+(t+.5)*cw)}' cy='{F(y0+(chemin[t]+.5)*ch)}' r='8' fill='none' stroke='#fff' stroke-width='2'/>"); }
for(int k=0;k<K;k++) fig.Append(Text(x0-10,y0+(k+.65)*ch,tags[k],"end"));
for(int j=0;j<6;j++) { int shade=240-185*j/5; fig.Append($"<rect x='{F(590+j*20)}' y='110' width='20' height='18' fill='#{shade:X2}{shade:X2}FF'/>"); }
fig.Append(Text(590,150,"0", "start")); fig.Append(Text(710,150,F(minDelta),"end")); fig.Append(Text(650,175,"écart log-score")); fig.Append(Text(650,215,"cercle : chemin")); fig.Append("</svg>"); display(new PlotSvg(fig.ToString()));

Phrase : le matelot voile la voile . ; Viterbi : DET NOM VER DET NOM PUNCT


le        DET -3.16 <- — | PRON -5.10 <- —


matelot   NOM -8.24 <- DET | VER -10.53 <- PRON


voile     VER -12.92 <- NOM | PUNCT -14.07 <- NOM


la        DET -15.45 <- VER | ADV -19.24 <- VER


voile     NOM -19.42 <- DET | VER -22.84 <- DET


.         PUNCT -21.76 <- NOM | ADJ -25.41 <- NOM


Treillis Viterbi : écart au meilleur état par position le matelot voile la voile . DET NOM VER ADJ PREP PRON ADV PUNCT 0 -6.541 écart log-score cercle : chemin

### Lecture — lire le treillis et ses backpointers

Le tableau texte donne, position par position, les deux états dominants et leur **prédécesseur gagnant** : on y lit la competition `NOM` contre `VER` aux positions de *voile*, et le détachement progressif du chemin optimal. La figure rend le même treillis visible : l'échelle est **l'écart au meilleur état de chaque position** (le maximum des $\delta$ de chaque colonne étant non croissant), et les cercles blancs tracent le chemin de Viterbi reconstruit par les backpointers. Le chemin global reconstruit peut passer par un état qui n’est pas le maximum local de sa colonne ; les backpointers relient les choix compatibles. Les positions ambiguës se reconnaissent à leurs scores proches.

## 5. Le cas non dégénéré — là où le contexte renverse la décision

Toute la démonstration tient dans deux phrases minimales :

| Phrase | Décideur token seul | Viterbi |
|---|---|---|
| *la porte verte* | `porte` → VER (émission dominante) | `porte` → **NOM** (car `DET → NOM`) |
| *elle porte une robe* | `porte` → VER | `porte` → **VER** (car `PRON → VER`) |

Le **même mot**, deux sorties différentes — pilotées par l'état précédent. Ce n'est pas un cas dégénéré fabriqué pour la cause : les paires minimales sont dans le test, et la mesure ci-dessous énumère **tous** les tokens où l'argmax des émissions seules et le décodage Viterbi divergent, avec le contexte autour, l'étiquette d'or, et le verdict.

In [9]:
string[] Emissions(string[] words) => words.Select(w=>tags[Enumerable.Range(0,K).OrderByDescending(i=>B[i,Wi(w)]).First()]).ToArray();
int flips=0,versOr=0;
for(int s=0;s<test.Length;s++) { var sent=test[s]; var w=sent.Select(x=>x.Word).ToArray(); var em=Emissions(w); var path=Decode(w).Item1;
 for(int j=0;j<w.Length;j++) if(em[j]!=tags[path[j]]) { flips++; bool bon=tags[path[j]]==sent[j].Tag; if(bon) versOr++; Console.WriteLine($"test[{s}] {w[j],-10} émission={em[j],-5} Viterbi={tags[path[j]],-5} or={sent[j].Tag,-5} {(bon?"CORRIGÉ":"PERDU")} [{string.Join(" ",w.Skip(Math.Max(0,j-1)).Take(3))}]"); } }
Console.WriteLine($"Total : {flips} renversements, dont {versOr} vers l'or");

test[0] voile      émission=VER   Viterbi=NOM   or=NOM   CORRIGÉ [la voile rouge]


test[0] tourne     émission=ADV   Viterbi=PUNCT or=VER   PERDU [rouge tourne .]


test[1] robe       émission=ADV   Viterbi=NOM   or=NOM   CORRIGÉ [une robe bleue]


test[2] vérité     émission=ADV   Viterbi=NOM   or=NOM   CORRIGÉ [la vérité .]


test[3] garde      émission=VER   Viterbi=NOM   or=NOM   CORRIGÉ [le garde ferme]


test[3] grille     émission=ADV   Viterbi=NOM   or=NOM   CORRIGÉ [la grille .]


... (8 outputs left out)

test[8] voile      émission=VER   Viterbi=NOM   or=NOM   CORRIGÉ [la voile .]


test[9] concierge  émission=ADV   Viterbi=NOM   or=NOM   CORRIGÉ [le concierge ouvre]


test[9] porte      émission=VER   Viterbi=NOM   or=NOM   CORRIGÉ [la porte verte]


Total : 17 renversements, dont 12 vers l'or


### Lecture — le contexte change réellement la décision, et souvent pour le mieux

La sortie énumère **tous** les renversements — chacun est un token où la **même** information locale (l'émission) existe, mais où la chaîne de transitions impose un autre état. Les inconnus dominent la liste (*robe*, *matelot*, *vérité*…) : l'émission d'un inconnu est quasi uniforme, donc **inutilisable seule** — c'est le contexte (`une _`, `le _`) qui les résout, presque toujours vers l'or (`CORRIGÉ`). Les mots ambigus y figurent aussi : *la voile rouge*, *le garde ferme*, *la coupe brille*, *une avance rapide*, le second *voile* du *matelot*, et *la porte verte* — **six** renversements VER→NOM, tous vers l'or, exactement là où l'émission se trompait. Le verdict `PERDU` signifie que le nouvel état est faux, sans nécessairement dire que l’ancien était correct. La colonne montre aussi ces erreurs : le renversement est une **règle de décision globale**, pas une garantie locale — le cas *il la garde* (section 9) y perd. Le bilan net se mesure en section 6.

## 6. La baseline indépendante par token — le plancher à battre

La baseline décide chaque token **indépendamment** de ses voisins : pour un mot vu au train, l'étiquette **majoritaire** de ses occurrences ; pour un inconnu, l'étiquette majoritaire **globale**. C'est le « tag le plus fréquent » des manuels — le même plancher que la baseline token-wise du [03_CRF_Etiquetage_Sequentiel](03_CRF_Etiquetage_Sequentiel.ipynb). Trois systèmes montent sur le banc : la baseline, les émissions seules du HMM (section 5), et le Viterbi complet.

In [10]:
var countsByWord=train.SelectMany(s=>s).GroupBy(x=>x.Word).ToDictionary(g=>g.Key,g=>g.GroupBy(x=>x.Tag).OrderByDescending(h=>h.Count()).First().Key);
string global=train.SelectMany(s=>s).GroupBy(x=>x.Tag).OrderByDescending(g=>g.Count()).First().Key;
string[] Baseline(string[] words) => words.Select(w=>countsByWord.GetValueOrDefault(w,global)).ToArray();
int CountCorrect(Func<string[],string[]> fn) => test.Sum(s=>{ var pred=fn(s.Select(x=>x.Word).ToArray()); return pred.Where((x,i)=>x==s[i].Tag).Count(); });
var counts=new[]{CountCorrect(Baseline),CountCorrect(Emissions),CountCorrect(w=>Decode(w).Item1.Select(i=>tags[i]).ToArray())}; int totalTest=test.Sum(x=>x.Length);
int differences=0;
foreach(var sent in test) { var words=sent.Select(x=>x.Word).ToArray(); var a=Baseline(words); var b=Emissions(words); differences+=a.Where((v,i)=>v!=b[i]).Count(); }
Console.WriteLine($"Désaccords baseline / émissions : {differences}/{totalTest} tokens");
string[] names={"Majorité/mot","Émissions","Viterbi"}; for(int i=0;i<3;i++) Console.WriteLine($"{names[i],-15} {counts[i]}/{totalTest} = {(double)counts[i]/totalTest:F3}");
Bars("Trois règles de décision sur le test","accuracy",names,new[]{counts.Select(x=>(double)x/totalTest).ToArray()},new[]{"Accuracy par token"},new[]{"#2ca02c"},1);

Désaccords baseline / émissions : 8/53 tokens


Majorité/mot    38/53 = 0.717


Émissions       38/53 = 0.717


Viterbi         47/53 = 0.887


Trois règles de décision sur le test 0 0.2 0.4 0.6 0.8 1 Majorité/mot Émissions Viterbi accuracy Accuracy par token

### Lecture — trois niveaux mesurés

La baseline et les émissions seules obtiennent chacune **38/53**, mais leurs prédictions ne sont pas identiques : elles divergent sur les huit inconnus, pour lesquels le repli majoritaire donne `DET` et l'émission lissée favorise `ADV`. Ces deux choix sont faux sur les huit tokens, d'où le même compte. En général l'argmax de $P(w|t)$ et le tag majoritaire d'un mot diffèrent aussi sur des mots connus, car l'émission est normalisée par les effectifs de chaque étiquette.

Le Viterbi complet atteint **47/53**, soit environ **17 points** de plus sur ce test. C'est une mesure sur ce corpus, pas une garantie générale. Le triptyque baseline / émissions / séquence permet de lire ce que les transitions changent avant de comparer les témoins.

## 7. Témoin bibliothèque I — hmmlearn, les mêmes paramètres rechargés

Premier témoin : le paquet `hmmlearn` fournit `CategoricalHMM`, un HMM à émissions catégorielles. Nous l'alimentons avec **nos** paramètres estimés ($\pi$, $A$, $B$ telles quelles) — il n'apprend rien. Ses deux méthodes clés implémentent **indépendamment** nos deux algorithmes : `score` (forward : $\log P(W)$, la somme des chemins) et `predict` (Viterbi : le chemin optimal). La concordance attendue est **exacte** — même modèle, mêmes calculs, autre code : c'est un étalonnage d'implémentation, pas de modélisation.

In [11]:
using(Py.GIL()) {
 dynamic np=Py.Import("numpy"); dynamic hmm=Py.Import("hmmlearn.hmm");
 PyList Row(IEnumerable<double> xs) { var r=new PyList(); foreach(double x in xs) { using var v=new PyFloat(x); r.Append(v); } return r; }
 PyList Matrix(double[,] m) { var r=new PyList(); for(int i=0;i<m.GetLength(0);i++) { using var row=Row(Enumerable.Range(0,m.GetLength(1)).Select(j=>m[i,j])); r.Append(row); } return r; }
 dynamic model=hmm.CategoricalHMM(n_components:K,init_params:""); using var pp=Row(pi); using var aa=Matrix(A); using var bb=Matrix(B);
 model.startprob_=np.array(pp); model.transmat_=np.array(aa); model.emissionprob_=np.array(bb);
 double maxDiff=0; int mismatch=0;
 foreach(var sent in test) { var words=sent.Select(x=>x.Word).ToArray(); var values=new double[words.Length,1]; for(int i=0;i<words.Length;i++) values[i,0]=Wi(words[i]);
  using var rows=Matrix(values); dynamic x=np.array(rows,dtype:np.int64); double score=(double)model.score(x); using PyObject p=model.predict(x); var decoded=p.InvokeMethod("tolist").As<int[]>();
  maxDiff=Math.Max(maxDiff,Math.Abs(score-ForwardWords(words))); mismatch+=decoded.Where((v,i)=>v!=Decode(words).Item1[i]).Count(); }
 Console.WriteLine($"Témoin hmmlearn : écart max forward {maxDiff:E3} ; désaccords de chemin {mismatch}");
}

Témoin hmmlearn : écart max forward 0.000E+000 ; désaccords de chemin 0


<null>

### Lecture — concordance numérique et de chemin avec le témoin

Sur cette exécution, l'écart maximal affiché entre le `score` de hmmlearn et notre forward est **0** en double précision ; `predict` et Viterbi ne divergent sur aucun des 53 tokens du test. Le témoin a reçu les mêmes paramètres $\pi$, $A$ et $B$, sans réapprentissage. C'est un contrôle croisé sur ces dix phrases, pas une démonstration de correction pour toute entrée.

## Exercice 1 — backward et décodage marginal (MPM)

Viterbi maximise la probabilité du **chemin entier** ; rien ne garantit que chaque position, prise isolément, y soit la plus probable. Le décodage **marginal** (maximum *a posteriori* par position) choisit à chaque position $i$ l'état $\arg\max_k P(t_i = k \mid W)$ — le chemin qui **minimise le nombre d'erreurs attendu par token**, au prix de possibles suites localement improbables.

**Objectif** : implémenter `Backward` (la récurrence symétrique du forward, $\beta$), puis `DecodeMpm` qui combine la table forward $\alpha$ et $\beta$ (backward), normalise, et prend l'argmax par position. Comparer sur le test au chemin de Viterbi : combien de tokens divergent, et dans quel sens ?

- **Étape 0** — écrire `ForwardTable`, une variante de `ForwardWords` qui **conserve la table $\alpha$ complète** — `ForwardWords` ne retourne que le scalaire final $\log P(W)$ — et vérifier que la somme log des états de la dernière ligne de `ForwardTable(ws)` redonne `ForwardWords(ws)` ;
- **Étape 1** — initialiser $\beta_{T-1}(k) = 0$ pour tout $k$ (log de 1) ;
- **Étape 2** — remonter : $\beta_t(k) = \log\sum_j e^{\,\log A_{kj} + \log B_j(w_{t+1}) + \beta_{t+1}(j)}$ ;
- **Étape 3** — $\log P(t_i = k \mid W) = \alpha_i(k) + \beta_i(k) - \log P(W)$, puis `argmax` par position — où $\alpha_i(k)$ est la **somme** forward, et non le max de Viterbi $\delta_i(k)$ : confondre les deux tables fausserait les postérieures, et c'est précisément ce que cet exercice apprend à distinguer.

> **Indice** : `LogSumExp` sert exactement comme dans `ForwardWords` ; et `log P(W)` est déjà la valeur retournée par `ForwardWords`.

In [12]:
// Exercice 1 : table forward complète, backward et décodage marginal.
double[,] ForwardTable(string[] words) { /* TODO : somme des chemins vers chaque état. */ return null; }
double[,] Backward(string[] words) { /* TODO : somme des suffixes depuis chaque état. */ return null; }
int[] DecodeMpm(string[] words) { /* TODO : combiner alpha et beta, puis argmax. */ return null; }
bool ValiderMpm(string[] words) {
    var alpha=ForwardTable(words); var beta=Backward(words); var path=DecodeMpm(words); int n=words.Length;
    if(alpha==null || beta==null || path==null || alpha.GetLength(0)!=n || alpha.GetLength(1)!=K || beta.GetLength(0)!=n || beta.GetLength(1)!=K || path.Length!=n || !alpha.Cast<double>().All(double.IsFinite) || !beta.Cast<double>().All(double.IsFinite) || path.Any(x=>x<0 || x>=K)) return false;
    var indices=words.Select(Wi).ToArray(); var marginal=new double[n,K]; var chain=new int[n]; double total=0;
    void Paths(int t,double weight) {
        if(t==n) { total+=weight; for(int u=0;u<n;u++) marginal[u,chain[u]]+=weight; return; }
        for(int k=0;k<K;k++) { chain[t]=k; Paths(t+1,weight*(t==0?pi[k]:A[chain[t-1],k])*B[k,indices[t]]); }
    }
    Paths(0,1);
    for(int t=0;t<n;t++) for(int state=0;state<K;state++) {
        int pos=t, target=state;
        double Prefix(int u,int prev,double weight) {
            if(u>pos) return prev==target?weight:0;
            double sum=0; for(int k=0;k<K;k++) sum+=Prefix(u+1,k,weight*(u==0?pi[k]:A[prev,k])*B[k,indices[u]]); return sum;
        }
        double Suffix(int u,int prev,double weight) {
            if(u==n) return weight;
            double sum=0; for(int k=0;k<K;k++) sum+=Suffix(u+1,k,weight*A[prev,k]*B[k,indices[u]]); return sum;
        }
        if(Math.Abs(alpha[t,state]-Math.Log(Prefix(0,0,1)))>1e-9 || Math.Abs(beta[t,state]-Math.Log(Suffix(t+1,state,1)))>1e-9) return false;
    }
    for(int t=0;t<n;t++) { int tt=t; int expected=Enumerable.Range(0,K).OrderByDescending(k=>marginal[tt,k]).First(); if(path[t]!=expected) return false; }
    return Math.Abs(Math.Log(total)-ForwardWords(words))<1e-9;
}
var ex1=DecodeMpm(new[]{"la","porte"});
if(ex1==null) Console.WriteLine("Exercice 1 à compléter");
else Console.WriteLine(ValiderMpm(new[]{"la","porte"}) && ValiderMpm(new[]{"il","la","garde"}) ? "Fixtures alpha, beta et MPM validées par énumération" : "Réponse incorrecte");

Exercice 1 à compléter


## 8. Témoin bibliothèque II — nltk, un pipeline complet sans lissage

Second témoin : `nltk.tag.hmm.HiddenMarkovModelTrainer` entraîne **de bout en bout** un HMM supervisé sur notre train — estimation **et** décodage lui appartiennent. À la différence de notre estimation, celle de NLTK est un MLE **sans lissage** : le moindre mot absent du train rend son émission nulle dans chaque état, et le chemin qui le traverse disparaît. Notre test contient 8 types inconnus (section 1) : la prédiction est mesurable, pas devinée. Note d'exécution : NLTK convertit ses log-probabilités nulles ($-\infty$) en cast `float32` interne, ce qui déclenche des `RuntimeWarning: overflow encountered in cast` — bénins pour le chemin retourné, ignorés ci-dessous pour la lisibilité.

In [13]:
using(Py.GIL()) {
 dynamic nh=Py.Import("nltk.tag.hmm"); dynamic warnings=Py.Import("warnings"); dynamic builtins=Py.Import("builtins");
 using var training=new PyList();
 foreach(var sent in train) { using var row=new PyList(); foreach(var (word,tag) in sent) { using var w=new PyString(word); using var t=new PyString(tag); using var pair=new PyTuple(new PyObject[]{w,t}); row.Append(pair); } training.Append(row); }
 dynamic scope=warnings.catch_warnings(); scope.__enter__(); int correctNltk=0,agreement=0,total=0;
 try {
  warnings.filterwarnings("ignore",message:"overflow encountered in cast",category:builtins.RuntimeWarning);
  dynamic trainer=nh.HiddenMarkovModelTrainer(); dynamic tagger=trainer.train_supervised(training);
  foreach(var sent in test) { using var words=new PyList(); foreach(var token in sent) { using var word=new PyString(token.Word); words.Append(word); }
   using PyObject result=tagger.tag(words); int[] ours=Decode(sent.Select(x=>x.Word).ToArray()).Item1; int i=0;
   for(i=0;i<sent.Length;i++) { using var pair=result.GetItem(i); using var tag=pair.GetItem(1); string value=tag.As<string>(); if(value==sent[i].Tag) correctNltk++; if(value==tags[ours[i]]) agreement++; total++; }
  }
 } finally { scope.__exit__(null,null,null); }
 Console.WriteLine($"Témoin NLTK sans lissage : {correctNltk}/{total} corrects ; accord avec Viterbi {agreement}/{total}");
}
var oovSentences=test.Select((s,i)=>(s,i)).Where(x=>x.s.Any(t=>!trainWords.Contains(t.Word))).ToArray();
foreach(var x in oovSentences) Console.WriteLine($"test[{x.i}] {string.Join(" ",x.s.Select(t=>t.Word))} : inconnus {string.Join(", ",x.s.Select(t=>t.Word).Where(w=>!trainWords.Contains(w)))}");
Console.WriteLine($"MLE non lissé : {oovSentences.Length}/{test.Length} phrases contiennent un inconnu");

Témoin NLTK sans lissage : 26/53 corrects ; accord avec Viterbi 23/53


test[0] la voile rouge tourne . : inconnus tourne


test[1] elle porte une robe bleue . : inconnus robe


test[2] il voile la vérité . : inconnus vérité


test[3] le garde ferme la grille . : inconnus grille


test[4] elle coupe le gazon . : inconnus gazon


test[7] une avance rapide surprend . : inconnus rapide


test[8] le matelot voile la voile . : inconnus matelot


test[9] le concierge ouvre la porte verte . : inconnus concierge


MLE non lissé : 8/10 phrases contiennent un inconnu


### Lecture — l'effondrement du non-lissage, et ses limites d'interprétation

Là où notre HMM lissé décode proprement, le pipeline NLTK **sans lissage** s'effondre : accuracy moitié moindre, accord partiel avec nos chemins. Le contrôle ci-dessus ancre le mécanisme sur **notre propre pipeline** : huit des dix phrases du test contiennent un type inconnu, et en MLE pur chacune a une vraisemblance strictement nulle — c'est le $\log 0$ de la section 2 à l'œuvre.

Nuance d'honnêteté : ce contraste est **illustratif**, pas une isolation causale stricte. Le score exact de NLTK dépend aussi de sa convention de repli sur les chemins de vraisemblance nulle (des suites guidées par les seuls tokens connus, différentes des nôtres) ; ce que le contraste isole, c'est la **cause dominante commune** — l'absence de lissage. Ce témoin n'est pas un contre-exemple pour NLTK : son API documente ce choix et fournit des estimateurs lissables ; c'est le même contraste que le MLE nu contre Laplace dans le [02_NGrammes_Modeles_De_Langue](02_NGrammes_Modeles_De_Langue.ipynb).

## Exercice 2 — le poids du lissage : balayage de α

Le add-1 de la section 2 est un choix, pas une nécessité. Que gagne-t-on ou perd-on à l'adoucir (α = 0.01) ou à le durcir (α = 5) ?

**Objectif** : faire varier α **des émissions uniquement**, ré-estimer $B$, redécoder le test par Viterbi, et rapporter l'accuracy token pour chaque α.

- **Étape 1** — factoriser l'estimation : recompter les émissions du train puis reconstruire `logB` (et seulement elle) pour chaque α ;
- **Étape 2** — pour chaque α, décoder les 10 phrases du test et compter les tokens corrects ;
- **Étape 3** — afficher la courbe `accuracy(α)` et comparer les valeurs à α = 1 ; ce test est descriptif, choisir α pour généraliser demande une validation séparée.

> **Indice** : les transitions et π restent celles du train (α_t inchangé) ; seules les émissions bougent — et avec elles le sort des 8 inconnus du test.

In [14]:
// Exercice 2 : réestimer les seules émissions pour plusieurs valeurs alpha.
Dictionary<double,double> BalayageAlpha(double[] alphas) { /* TODO : figer pi/A, reconstruire B et mesurer le test. */ return null; }
var alphasEx=new[]{0.01,0.1,0.5,1.0,5.0}; var ex2=BalayageAlpha(alphasEx);
if(ex2==null) Console.WriteLine("Exercice 2 à compléter");
else {
    var attendus=new[]{50,49,47,47,44};
    bool valide=ex2.Count==alphasEx.Length && alphasEx.Select((a,i)=>ex2.TryGetValue(a,out double v) && double.IsFinite(v) && Math.Abs(v-(double)attendus[i]/totalTest)<1e-9).All(x=>x);
    var autres=BalayageAlpha(new[]{0.2,2.0});
    valide=valide && autres!=null && autres.Count==2 && autres.TryGetValue(0.2,out var a) && autres.TryGetValue(2.0,out var b) && double.IsFinite(a) && double.IsFinite(b) && Math.Abs(a-47.0/53)<1e-9 && Math.Abs(b-44.0/53)<1e-9;
    Console.WriteLine(valide ? "Fixtures du balayage validées" : "Réponse incorrecte");
}

Exercice 2 à compléter


## 9. Analyse d'erreurs — six tokens résistent

Aucune métrique ne vaut la lecture des erreurs une à une — la discipline du [03_CRF_Etiquetage_Sequentiel](03_CRF_Etiquetage_Sequentiel.ipynb), appliquée à notre Viterbi.

In [15]:
int erreurs=0; for(int i=0;i<test.Length;i++) { var s=test[i]; var w=s.Select(x=>x.Word).ToArray(); var path=Decode(w).Item1; for(int j=0;j<w.Length;j++) if(tags[path[j]]!=s[j].Tag) { erreurs++; Console.WriteLine($"test[{i}] {w[j],-10} prédit={tags[path[j]],-5} or={s[j].Tag,-5} [{string.Join(" ",w.Skip(Math.Max(0,j-2)).Take(5))}]"); } }
Console.WriteLine($"Total : {erreurs}/{totalTest} tokens en erreur");

test[0] tourne     prédit=PUNCT or=VER   [voile rouge tourne .]


test[5] brille     prédit=ADJ   or=VER   [la coupe brille .]


test[6] la         prédit=DET   or=PRON  [il la garde .]


test[6] garde      prédit=NOM   or=VER   [il la garde .]


test[7] rapide     prédit=VER   or=ADJ   [une avance rapide surprend .]


test[7] surprend   prédit=ADV   or=VER   [avance rapide surprend .]


Total : 6/53 tokens en erreur


### Lecture — trois familles d'erreurs, une leçon

Les erreurs se groupent en familles (sortie ci-dessus) : le **clitique objet** (*il la garde* : notre chaîne préfère `PRON DET` à `PRON PRON`, car le train n'offre que quatre clitiques contre des dizaines de `la` déterminants) ; l'**inconnu verbal** (*tourne* : l'émission d'inconnu pointe vers l'état rare `ADV`, piège documenté en section 2, et la transition qui suit ne suffit pas toujours à le rattraper) ; et l'**alternance ADJ/VER en fin de syntagme** (*coupe brille* : après un nom, `NOM ADJ .` et `NOM VER .` sont tous deux plausibles dans ce petit train). Plusieurs pistes restent à mesurer : ajouter des clitiques au train, adapter le traitement des inconnus ou utiliser des poids de transition **conditionnés aux mots**, comme dans le CRF du notebook 03. L'exercice 2 fait varier le lissage des émissions ; l'exercice 3 ajoute le mot ambigu *couche* au train. Aucun ne démontre ici la correction des clitiques.

## Exercice 3 — étendre le corpus : le mot « couche »

Ajouter des phrases d'entraînement déplace les comptages MLE. Cet exercice étudie le mot ambigu *couche* ; il ne mesure pas à lui seul une correction des erreurs de clitiques.

**Objectif** : étendre le corpus avec le mot ambigu *couche* — nom (*une couche épaisse*) et verbe (*la nourrice couche les enfants*) — puis ré-estimer et mesurer ce qui change **et ce qui ne change pas**.

- **Étape 1** — écrire 4 à 6 phrases couvrant les deux sens, dans le style des blocs du corpus (section 1) ;
- **Étape 2** — les ajouter au **train** (jamais au test), ré-estimer $\pi, A, B$ à partir des comptages ;
- **Étape 3** — décoder une phrase-test contenant *couche* en contexte minoritaire, et rapporter : le chemin obtenu, la réponse de l'argmax d'émissions, et l'effet sur l'accuracy du test d'origine.

> **Indice** : les paramètres de l'HMM ne « savent » rien du mot *couche* avant vos phrases — chaque sens doit apparaître au moins deux fois pour lutter contre le contexte.

In [16]:
// Exercice 3 : les exemples fournis sont ajoutés au TRAIN seulement.
(string[] Chemin,string[] Emissions,double Accuracy)? EtendreAvecCouche(
    (string Word,string Tag)[][] ajout, string[] phrase) {
    // TODO : réestimer avec train + ajout, décoder phrase et mesurer le TEST initial.
    return null;
}
var ajoutEx=new (string Word,string Tag)[][] {
    new[]{("la","DET"),("couche","NOM"),("épaisse","ADJ"),(".","PUNCT")},
    new[]{("une","DET"),("couche","NOM"),("blanche","ADJ"),(".","PUNCT")},
    new[]{("elle","PRON"),("couche","VER"),("le","DET"),("chat","NOM"),(".","PUNCT")},
    new[]{("il","PRON"),("couche","VER"),("la","DET"),("souris","NOM"),(".","PUNCT")}
};
var ex3=EtendreAvecCouche(ajoutEx,new[]{"la","couche","."});
if(ex3==null) Console.WriteLine("Exercice 3 à compléter");
else {
 var autre=EtendreAvecCouche(ajoutEx,new[]{"elle","couche","le","chat","."});
 bool valide=autre.HasValue && ex3.Value.Chemin!=null && ex3.Value.Emissions!=null && autre.Value.Chemin!=null && autre.Value.Emissions!=null &&
  ex3.Value.Chemin.SequenceEqual(new[]{"DET","NOM","PUNCT"}) && ex3.Value.Emissions.SequenceEqual(new[]{"DET","VER","PUNCT"}) &&
  autre.Value.Chemin.SequenceEqual(new[]{"PRON","VER","DET","NOM","PUNCT"}) && autre.Value.Emissions.SequenceEqual(new[]{"PRON","VER","DET","NOM","PUNCT"}) &&
  double.IsFinite(ex3.Value.Accuracy) && double.IsFinite(autre.Value.Accuracy) && Math.Abs(ex3.Value.Accuracy-47.0/53)<1e-9 && Math.Abs(autre.Value.Accuracy-47.0/53)<1e-9;
 var ajout2=Enumerable.Repeat(ajoutEx[0],4).Concat(Enumerable.Repeat(ajoutEx[2],2)).ToArray();
 var autreCorpus=EtendreAvecCouche(ajout2,new[]{"la","couche","."});
 valide=valide && autreCorpus.HasValue && autreCorpus.Value.Chemin!=null && autreCorpus.Value.Emissions!=null &&
  autreCorpus.Value.Chemin.SequenceEqual(new[]{"DET","NOM","PUNCT"}) && autreCorpus.Value.Emissions.SequenceEqual(new[]{"DET","NOM","PUNCT"}) &&
  double.IsFinite(autreCorpus.Value.Accuracy) && Math.Abs(autreCorpus.Value.Accuracy-47.0/53)<1e-9;
 Console.WriteLine(valide ? "Fixtures extension validées" : "Réponse incorrecte");
}

Exercice 3 à compléter


## À retenir
**La séquence comme preuve, la mesure comme juge.**


**Ce que ce notebook a construit** — et que la série ne portait pas avant lui :

1. **Un HMM d'étiquetage morphosyntaxique complet from scratch** : estimation MLE des transitions et émissions sur un corpus annoté, lissage add-1 avec symbole d'inconnu, forward en log-espace (`LogSumExp`), décodage Viterbi avec backpointers et reconstruction à l'envers — le tout en C# sur CPU et déterministe, avec Python.NET pour les seuls témoins ;
2. **Un corpus POS français synthétique à provenance explicite** — rédigé pour ce notebook, pas un benchmark — concentrant deux couches d'ambiguïté mesurables : cinq mots NOM/VER et trois clitiques DET/PRON, jusqu'à la phrase à double lecture *le matelot voile la voile*, décodée correctement ;
3. **La démonstration non dégénérée du contexte de séquence** : les renversements token par token entre l'argmax d'émissions et le Viterbi complet sont énumérés avec leur contexte et leur verdict — la même forme *porte* est étiquetée deux fois différemment selon la chaîne ;
4. **Trois niveaux mesurés sur le même banc** : baseline majoritaire par mot, émissions seules, Viterbi complet — l'hypothèse markovienne vaut ici environ dix-sept points d'accuracy ;
5. **Deux témoins bibliothèque réellement exécutés** : `hmmlearn` chargé avec nos paramètres reproduit forward (`score`) et Viterbi (`predict`) **à l'identique** (écart machine, zéro divergence) ; `nltk` entraîné sans lissage s'effondre sur les inconnus — la mesure de ce que le add-1 achète ;
6. **Une analyse d'erreurs lue une à une** : clitiques objets, inconnus verbaux, alternance ADJ/VER — trois familles qui suggèrent de comparer davantage de données, le lissage des émissions et des poids conditionnels comme ceux du CRF ; les exercices 2 et 3 mesurent respectivement un balayage du lissage et un ajout de phrases avec *couche*.

**Étape suivante** : le [03_CRF_Etiquetage_Sequentiel](03_CRF_Etiquetage_Sequentiel.ipynb) garde le treillis et le Viterbi, mais **réécrit la paramétrisation** — scores d'émission et de transition appris discriminativement, conditionnellement aux mots et à leurs traits, par maximisation de la vraisemblance conditionnelle. Le génératif et le discriminatif, même banc d'expérience.

## Vérifiez votre compréhension

1. Le banc mesure trois niveaux (classe majoritaire par mot, émissions seules, Viterbi complet) : combien vaut l'hypothèse markovienne en points d'accuracy ?
2. Que reproduit exactement le témoin `hmmlearn`, et que démontre au contraire l'effondrement du témoin NLTK sans lissage ?
3. Quelles pistes suggèrent les trois familles d’erreurs, et que mesurent exactement les exercices 2 et 3 ?
4. Dans le décodage marginal (exercice 1), pourquoi la table forward α et non le max de Viterbi δ ?

<details><summary>Réponses</summary>

1. Environ **dix-sept points** d'accuracy sur ce corpus : l'écart mesuré entre décoder token par token et décoder la séquence entière.
2. `hmmlearn`, rechargé avec **nos** paramètres, redonne forward (`score`) et Viterbi (`predict`) **à l'identique** — écart machine, zéro divergence. NLTK entraîné en MLE pur rend une émission nulle pour chaque mot inconnu : huit phrases de test sur dix ont alors une vraisemblance strictement nulle — la mesure de ce que le add-1 achète.
3. Ajouter des clitiques au train ou conditionner les transitions aux mots sont des pistes à évaluer. L'exercice 2 varie le lissage des émissions ; l'exercice 3 ajoute des exemples de *couche*. Aucun des deux ne démontre que les erreurs de clitiques sont corrigées.
4. Parce que le postérieur `P(t_i = k | W)` somme sur tous les chemins qui passent par l'état `k` à la position `i` — c'est la table α — quand δ n'en garde que le meilleur : confondre les deux tables fausse les postérieures.

</details>


## Pour aller plus loin

**Carnet suivant** : [06-Recherche-Information-TFIDF-BM25.ipynb](06-Recherche-Information-TFIDF-BM25.ipynb) — la même arithmétique de probabilités et de fréquences, tournée vers l'utilisateur : classer des documents.

- **Lectures** : Rabiner 1989, « A Tutorial on Hidden Markov Models and Selected Applications in Speech Recognition » ; Jurafsky & Martin, ch. « Sequence Labeling » et l'annexe HMM.
- **Dans le dépôt** : [03_CRF_Etiquetage_Sequentiel.ipynb](03_CRF_Etiquetage_Sequentiel.ipynb) — le même treillis et le même Viterbi, paramétrisation discriminative cette fois ; [02_NGrammes_Modeles_De_Langue.ipynb](02_NGrammes_Modeles_De_Langue.ipynb) pour le même contraste MLE/lissage.
